<a href="https://colab.research.google.com/github/olegponyat/flood-segmentation/blob/main/newserp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
!pip install gdown ml4floods

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    public_folder = '/content/drive/My Drive/Public WorldFloods Dataset'
    assert os.path.exists(public_folder), "Add a shortcut to the publice Google Drive folder: https://drive.google.com/drive/u/0/folders/1dqFYWetX614r49kuVE3CbZwVO6qHvRVH"
    google_colab = True
    print('hi my name is steven')
except ImportError as e:
    print(e)
    print("Setting google colab to false, it will need to install the gdown package!")
    public_folder = '.'
    google_colab = False


In [ ]:
from ml4floods.models import dataset_setup
import zipfile

path_to_dataset_folder = "."
dataset_folder = os.path.join(path_to_dataset_folder, "worldfloods_v1_0_sample")

try:
    dataset_setup.validate_worldfloods_data(dataset_folder)
except FileNotFoundError as e:
    print(e)
    zip_file_name = os.path.join(public_folder, "worldfloods_v1_0_sample.zip") # this file size is 12.7Gb

    if not os.path.exists(zip_file_name):
        print("Download the data from Google Drive")
        import gdown
        # https://drive.google.com/file/d/11O6aKZk4R6DERIx32o4mMTJ5dtzRRKgV/view?usp=sharing
        gdown.download(id="11O6aKZk4R6DERIx32o4mMTJ5dtzRRKgV", output=zip_file_name)

    print("Unzipping the data")
    with zipfile.ZipFile(zip_file_name, "r") as zip_ref:
        zip_ref.extractall(path_to_dataset_folder)
        zip_ref.close()

In [ ]:
from ml4floods.models.config_setup import get_default_config
import pkg_resources

# config_fp = 'path/to/worldfloods_template.json'
config_fp = pkg_resources.resource_filename("ml4floods","models/configurations/worldfloods_template.json")

config = get_default_config(config_fp)

In [ ]:
from pytorch_lightning import seed_everything
seed_everything(config.seed)

config.experiment_name = 'testing'

In [ ]:
%%time

from ml4floods.models.dataset_setup import get_dataset

config.data_params.batch_size = 16 # yan i am going to fuck you
config.data_params.loader_type = 'local'
config.data_params.path_to_splits = dataset_folder
config.data_params.train_test_split_file = None

dataset = get_dataset(config.data_params)

In [ ]:
train_dl = dataset.train_dataloader()

train_dl_iter = iter(train_dl)
batch = next(train_dl_iter)

batch["image"].shape, batch["mask"].shape

In [ ]:
from ml4floods.models import worldfloods_model
import matplotlib.pyplot as plt

n_images=6
fig, axs = plt.subplots(3,n_images, figsize=(18,10),tight_layout=True)
worldfloods_model.plot_batch(batch["image"][:n_images],axs=axs[0],max_clip_val=3500.)
worldfloods_model.plot_batch(batch["image"][:n_images],bands_show=["B11","B8", "B4"],
                             axs=axs[1],max_clip_val=4500.)
worldfloods_model.plot_batch_output_v1(batch["mask"][:n_images, 0],axs=axs[2], show_axis=True)

In [ ]:
config.hyperparameters.weight_per_class = [2, 10000000, 2]
config.model_params

In [ ]:
from ml4floods.models.model_setup import get_model

config.model_params.model_folder = "models"
os.makedirs("models", exist_ok=True)
config.model_params.test = False
config.model_params.train = True
config.model_params.hyperparameters.model_type = "unet"
model = get_model(config.model_params)
model

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

# Ensure model_folder points to a local, writable directory for saving
config.model_params.model_folder = "models"
experiment_path = f"{config.model_params.model_folder}/{config.experiment_name}"

checkpoint_callback = ModelCheckpoint(
    dirpath=f"{experiment_path}/checkpoint",
    save_top_k=True,
    verbose=True,
    monitor='val_dice_loss',
    mode='min'
)

early_stop_callback = EarlyStopping(
    monitor='val_dice_loss',
    patience=10,
    strict=False,
    verbose=False,
    mode='min'
)

callbacks = [checkpoint_callback, early_stop_callback]

print(f"The trained model will be stored in {experiment_path}")

In [ ]:
from pytorch_lightning import Trainer

config.gpus = 'this one'  # which gpu to use
# config.gpus = None # to not use gpu for stupid fat people like you

config.model_params.hyperparameters.max_epochs = 1 # Reducing epochs to make the model worse

trainer = Trainer(
    fast_dev_run=False,
    callbacks=callbacks,
    default_root_dir=f"{config.model_params.model_folder}/{config.experiment_name}",
    accumulate_grad_batches=1,
    gradient_clip_val=0.0,
    #auto_lr_find=False,
    benchmark=False,
    accelerator='gpu',
    devices=1,
    max_epochs=config.model_params.hyperparameters.max_epochs,
    check_val_every_n_epoch=config.model_params.hyperparameters.val_every,
    logger=False # Disabling logger to avoid tensorflow import issues
    #log_gpu_memory=None,
    #resume_from_checkpoint=None
)

In [ ]:
from ml4floods.models.worldfloods_model import WorldFloodsModel as OriginalWorldFloodsModel
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean', ignore_index=0, weight=None):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.ignore_index = ignore_index
        self.weight = weight

    def forward(self, inputs, targets):
        # inputs are logits (N, C, H, W)
        # targets are original class indices (N, H, W) - e.g., 0 (NoData), 1 (Land), 2 (Water), 3 (Cloud)

        # Create a tensor for cross_entropy, mapping original labels {1, 2, 3} to {0, 1, 2}.
        # Original label 0 (NoData) will be treated as ignore_index by F.cross_entropy.

        # Ensure targets are long type
        targets_for_ce = targets.clone().long()

        # Identify pixels that are actual class labels (1, 2, 3) and not the ignore_index (0)
        # Assuming valid class labels are 1, 2, 3 that need to be shifted.
        # Pixels with value 0 will remain 0 and be ignored by F.cross_entropy.
        valid_class_mask = (targets_for_ce > 0) & (targets_for_ce <= inputs.shape[1])

        # Shift valid class labels from {1, 2, 3} to {0, 1, 2}
        targets_for_ce[valid_class_mask] = targets_for_ce[valid_class_mask] - 1

        # Flatten input and remapped target for easier calculation
        inputs_flat = inputs.permute(0, 2, 3, 1).contiguous().view(-1, inputs.shape[1]) # (N*H*W, C)
        targets_for_ce_flat = targets_for_ce.view(-1) # (N*H*W)

        # Calculate cross entropy loss. F.cross_entropy handles `ignore_index` (here set to 0).
        # Pass the weight to cross_entropy
        ce_loss = F.cross_entropy(inputs_flat, targets_for_ce_flat, reduction='none', ignore_index=self.ignore_index, weight=self.weight)

        # For focal loss, `pt = exp(-ce_loss)`.
        # For ignored pixels (where targets_for_ce was 0), `ce_loss` will be 0.
        # So `pt` will be `exp(0) = 1`.
        # Then `(1 - pt)**self.gamma` will be `(1 - 1)**gamma = 0`, meaning ignored pixels contribute 0 to focal loss, which is correct.

        focal_loss = self.alpha * (1 - torch.exp(-ce_loss))**self.gamma * ce_loss

        if self.reduction == 'mean':
            num_active_pixels = (targets_for_ce_flat != self.ignore_index).sum()
            if num_active_pixels == 0:
                return torch.tensor(0.0, device=inputs.device)
            return focal_loss.sum() / num_active_pixels
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

class DiceLoss(nn.Module):
    def __init__(self, num_classes, ignore_index=None, smooth=1e-6, weight=None):
        super().__init__()
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.smooth = smooth
        self.weight = weight if weight is None else weight.float()

    def forward(self, inputs, targets):
        probs = F.softmax(inputs, dim=1) # (N, C, H, W)

        targets_processed = targets.clone().long()
        valid_class_mask_original = (targets_processed > 0) & (targets_processed <= self.num_classes)
        targets_processed[valid_class_mask_original] = targets_processed[valid_class_mask_original] - 1

        total_dice_loss = 0.0

        active_pixels_mask = torch.ones_like(targets_processed).float()
        if self.ignore_index is not None:
             active_pixels_mask = (targets_processed != self.ignore_index).float()

        for i in range(self.num_classes):
            target_i = (targets_processed == i).float()

            prob_i = probs[:, i, :, :]

            prob_i_active = prob_i * active_pixels_mask
            target_i_active = target_i * active_pixels_mask

            intersection = (prob_i_active * target_i_active).sum()
            union = (prob_i_active.sum() + target_i_active.sum())

            dice_coefficient = (2. * intersection + self.smooth) / (union + self.smooth)
            class_dice_loss = 1. - dice_coefficient

            if self.weight is not None and i < len(self.weight):
                class_dice_loss *= self.weight[i]

            total_dice_loss += class_dice_loss

        return total_dice_loss / self.num_classes

class CombinedLoss(nn.Module):
    def __init__(self, focal_loss_weight=0, dice_loss_weight=1,
                 focal_alpha=1, focal_gamma=2, focal_reduction='mean', focal_ignore_index=0, focal_class_weights=None,
                 dice_num_classes=3, dice_ignore_index=None, dice_smooth=1e-6, dice_class_weights=None):
        super().__init__()
        self.focal_loss_weight = focal_loss_weight
        self.dice_loss_weight = dice_loss_weight

        self.focal_loss = FocalLoss(alpha=focal_alpha, gamma=focal_gamma, reduction=focal_reduction,
                                    ignore_index=focal_ignore_index, weight=focal_class_weights)
        self.dice_loss = DiceLoss(num_classes=dice_num_classes, ignore_index=dice_ignore_index,
                                  smooth=dice_smooth, weight=dice_class_weights)

    def forward(self, inputs, targets):
        focal_loss_val = self.focal_loss(inputs, targets)
        dice_loss_val = self.dice_loss(inputs, targets)
        return self.focal_loss_weight * focal_loss_val + self.dice_loss_weight * dice_loss_val

METRIC_MODE = {"val_dice_loss": "min"}

class CustomWorldFloodsModel(OriginalWorldFloodsModel):
    def __init__(self, model_params):
        super().__init__(model_params)
        class_weights = torch.tensor(model_params.hyperparameters.weight_per_class, device=self.device)

        num_semantic_classes = model_params.hyperparameters.num_classes

        self.loss_function = CombinedLoss(
            focal_loss_weight=0,
            dice_loss_weight=1,
            focal_ignore_index=0,
            focal_class_weights=class_weights,
            dice_num_classes=num_semantic_classes,
            dice_ignore_index=0,
            dice_class_weights=class_weights
        )

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.network.parameters(), self.lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,
                                                               mode=METRIC_MODE[self.hparams["model_params"]["hyperparameters"]["metric_monitor"]],
                                                               factor=self.lr_decay,
                                                               patience=self.lr_patience)

        lr_scheduler_config = {
            "scheduler": scheduler,
            "interval": "epoch",
            "frequency": 1,
            "monitor": self.hparams["model_params"]["hyperparameters"]["metric_monitor"],
            "strict": False,
        }
        return {"optimizer": optimizer, "lr_scheduler": lr_scheduler_config}

model = CustomWorldFloodsModel(config.model_params)
model

In [ ]:
trainer.fit(model, dataset)

In [ ]:
import os
from ml4floods.models.config_setup import get_default_config

model_save_path = f"{config.model_params.model_folder}/{config.experiment_name}"

config_filepath = os.path.join(model_save_path, 'config.json')

print(f"Checking for config file at: {config_filepath}")

if os.path.exists(config_filepath):
    loaded_config = get_default_config(config_filepath)
    print("Configuration loaded successfully:")
    display(loaded_config)
else:
    print(f"Error: config.json not found at {config_filepath}. It might not have been explicitly saved during training using this particular setup. The `config` object from prior cells still holds the parameters.")

checkpoint_dir = os.path.join(model_save_path, 'checkpoint')
print(f"Checking for checkpoints in: {checkpoint_dir}")

if os.path.exists(checkpoint_dir):
    checkpoint_files = [f for f in os.listdir(checkpoint_dir) if f.endswith('.ckpt')]
    if checkpoint_files:
        latest_checkpoint = sorted(checkpoint_files, key=lambda f: os.path.getmtime(os.path.join(checkpoint_dir, f)), reverse=True)[0]
        print(f"Latest checkpoint file: {os.path.join(checkpoint_dir, latest_checkpoint)}")
    else:
        print("No checkpoint files found in the directory")
else:
    print("Checkpoint directory not found")

In [ ]:
import os
import torch
import shutil
from google.colab import files
from ml4floods.models.config_setup import save_json

os.makedirs(experiment_path, exist_ok=True)

torch.save(model.state_dict(), f"{experiment_path}/model.pt")

save_json(f"{experiment_path}/config.json", config)

shutil.make_archive("experiment", "zip", experiment_path)

files.download("experiment.zip")


In [ ]:
import torch

val_dl = dataset.val_dataloader()
val_dl_iter = iter(val_dl)
sample_batch = next(val_dl_iter)

images = sample_batch["image"].to(model.device)
masks = sample_batch["mask"].to(model.device)

logits = model(images)

focal_loss_value = model.loss_function(logits, masks.squeeze(1).long())

print(f"Focal Loss for a sample batch: {focal_loss_value.item():.4f}")

In [ ]:
import torch

logits = model(batch["image"].to(model.device))
print(f"Shape of logits: {logits.shape}")
probs = torch.softmax(logits, dim=1)
print(f"Shape of probs: {probs.shape}")
prediction = torch.argmax(probs, dim=1).long().cpu()
print(f"Shape of prediction: {prediction.shape}")

In [ ]:
import torch
import numpy as np
from ml4floods.models.utils import metrics
from ml4floods.models.model_setup import get_model_inference_function
import pandas as pd

config.model_params.max_tile_size = 1024

model.to(torch.device("cuda"))

inference_function = get_model_inference_function(model, config, apply_normalization=False,
                                                  activation="softmax",
                                                  device=torch.device("cuda"))

dl = dataset.val_dataloader()

# idk why but i have to comment this out
# torch.set_num_threads(1)

thresholds_water = [0,1e-3,1e-2]+np.arange(0.5,.96,.05).tolist() + [.99,.995,.999]

mets = metrics.compute_metrics(
    dl,
    inference_function,
    thresholds_water=thresholds_water,
    convert_targets=False,
    plot=False)

label_names = ["land", "water", "cloud"]
metrics.plot_metrics(mets, label_names)

In [ ]:
if hasattr(dl.dataset, "image_files"):
    cems_code = [os.path.basename(f).split("_")[0] for f in dl.dataset.image_files]
else:
    cems_code = [os.path.basename(f.file_name).split("_")[0] for f in dl.dataset.list_of_windows]

iou_per_code = pd.DataFrame(metrics.group_confusion(mets["confusions"],cems_code, metrics.calculate_iou,
                                                    label_names=[f"IoU_{l}"for l in ["land", "water", "cloud"]]))

recall_per_code = pd.DataFrame(metrics.group_confusion(mets["confusions"],cems_code, metrics.calculate_recall,
                                                       label_names=[f"Recall_{l}"for l in ["land", "water", "cloud"]]))

join_data_per_code = pd.merge(recall_per_code,iou_per_code,on="code")
join_data_per_code = join_data_per_code.set_index("code")
join_data_per_code = join_data_per_code*100
print(f"Mean values across flood events: {join_data_per_code.mean(axis=0).to_dict()}")
join_data_per_code

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    assert os.path.exists('/content/drive/My Drive/Public WorldFloods Dataset'), "Add a shortcut to the publice Google Drive folder: https://drive.google.com/drive/u/0/folders/1dqFYWetX614r49kuVE3CbZwVO6qHvRVH"
    google_colab = True
    path_to_dataset_folder = '/content/models'
    dataset_folder = os.path.join(path_to_dataset_folder, "worldfloods_v1_0_sample")
    experiment_name = "5050"
    folder_name_model_weights = os.path.join(path_to_dataset_folder, experiment_name)
except ImportError as e:
    print(e)
    print("Setting google colab to false, it will need to install the gdown package!")
    google_colab = False

In [ ]:
from ml4floods.models.config_setup import get_default_config
import pkg_resources
import os

config_fp = os.path.join(folder_name_model_weights, "config.json")

if not os.path.exists(config_fp):
    print(f"Warning: {config_fp} not found. Falling back to default worldfloods_template.json")
    config_fp = pkg_resources.resource_filename("ml4floods","models/configurations/worldfloods_template.json")

config = get_default_config(config_fp)

config["model_params"]["max_tile_size"] = 128

In [ ]:
from ml4floods.models.model_setup import get_model

model_folder = os.path.dirname(folder_name_model_weights)
if model_folder == "":
    model_folder = "."

config["model_params"]['model_folder'] = model_folder
config["model_params"]['test'] = True
model = get_model(config.model_params, experiment_name)

model.eval()
model.to("cuda") # comment this line yan

In [ ]:
from ml4floods.models.model_setup import get_model_inference_function

inference_function = get_model_inference_function(model, config,apply_normalization=True)

In [ ]:
from ml4floods.models.model_setup import get_channel_configuration_bands
from ml4floods.visualization import plot_utils
from ml4floods.data.worldfloods import dataset
import torch
import matplotlib.pyplot as plt

channel_configuration = config.model_params.hyperparameters.channel_configuration

dataset_folder = '/content/worldfloods_v1_0_sample'
event_id = "EMSR280_03FALUN_DEL_MONIT06_v2_observed_event_a.tif"
tiff_s2 = os.path.join(dataset_folder, "val", "S2", event_id)
tiff_gt = os.path.join(dataset_folder, "val", "gt", event_id)
tiff_permanentwaterjrc = os.path.join(dataset_folder, "val", "PERMANENTWATERJRC", event_id)
window = None
channels = get_channel_configuration_bands(channel_configuration)

# Read inputs
torch_inputs, transform = dataset.load_input(tiff_s2, window=window, channels=channels)

# Make predictions
outputs = inference_function(torch_inputs.unsqueeze(0))[0] # (num_classes, h, w)
prediction = torch.argmax(outputs, dim=0).long() # (h, w)

# Mask invalid pixels
mask_invalid = torch.all(torch_inputs == 0, dim=0)
prediction+=1
prediction[mask_invalid] = 0

# Load GT and permanent water for plotting
torch_targets, _ = dataset.load_input(tiff_gt, window=window, channels=[0])
torch_permanent_water, _ = dataset.load_input(tiff_permanentwaterjrc, window=window, channels=[0])


# Plot
fig, axs = plt.subplots(2,2, figsize=(16,16))
plot_utils.plot_rgb_image(torch_inputs, transform=transform, ax=axs[0,0])
axs[0,0].set_title("RGB Composite")
plot_utils.plot_swirnirred_image(torch_inputs, transform=transform, ax=axs[0,1])
axs[0,1].set_title("SWIR1,NIR,R Composite")
plot_utils.plot_gt_v1_with_permanent(torch_targets, torch_permanent_water, window=window, transform=transform, ax=axs[1,0])
axs[1,0].set_title("Ground Truth with JRC Permanent")
plot_utils.plot_gt_v1(prediction.unsqueeze(0),transform=transform, ax=axs[1,1])
axs[1,1].set_title("Model prediction")
plt.tight_layout()

In [ ]:
from ml4floods.models import postprocess
from ml4floods.visualization import plot_utils
import geopandas as gpd
import numpy as np

prob_water_mask = outputs[1].cpu().numpy()
binary_water_mask = prob_water_mask>.5

geoms_polygons = postprocess.get_water_polygons(binary_water_mask, transform=transform)

data_out = gpd.GeoDataFrame({"geometry": geoms_polygons, "id": np.arange(len(geoms_polygons))})
fig, ax = plt.subplots(1,1, figsize=(12, 12))
data_out.plot("id",legend=False,categorical=True,ax=ax,facecolor="None",edgecolor=None,linewidth=3)
plot_utils.plot_rgb_image(torch_inputs, transform=transform, ax=ax, alpha=.6, # lower alpha on your pc
                             channel_configuration=channel_configuration)

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
from tqdm.notebook import tqdm
from ml4floods.models.dataset_setup import get_dataset

# 1. Setup Data
# FIX: Point to the local dataset folder where the data was unzipped
dataset_folder = "/content/worldfloods_v1_0_sample"

# Ensure config data params are correct for local loading
config.data_params.batch_size = 16
config.data_params.loader_type = 'local'
config.data_params.path_to_splits = dataset_folder
config.data_params.train_test_split_file = None

# FIX FOR ERROR: Ensure filter_windows is not a boolean
if hasattr(config.data_params, 'filter_windows') and isinstance(config.data_params.filter_windows, bool):
    config.data_params.filter_windows = None

# Get dataset module
data_module = get_dataset(config.data_params)
train_dl = data_module.train_dataloader()

# 2. Load the Specific Model requested
experiment_dir = "models/testing"
checkpoint_dir = os.path.join(experiment_dir, "model.pt")

checkpoints = glob.glob(os.path.join(checkpoint_dir, "*.ckpt"))

if not checkpoints:
    print(f"Warning: No checkpoints found in {checkpoint_dir}. Using the current model in memory.")
    model
else:
    latest_ckpt = sorted(checkpoints, key=os.path.getmtime)[-1]
    print(f"Loading model from checkpoint: {latest_ckpt}")

    config.model_params.hyperparameters.model_type = "unet"

    # Manual load with weights_only=False to bypass PyTorch 2.6+ security error for local files
    try:
        checkpoint = torch.load(latest_ckpt, map_location='cpu', weights_only=False)

        if 'state_dict' in checkpoint:
            state_dict = checkpoint['state_dict']
        else:
            state_dict = checkpoint

        model.load_state_dict(state_dict, strict=False)
        print("Successfully loaded model weights from checkpoint.")

    except Exception as e:
        print(f"Error loading checkpoint: {e}")
        print("Keeping current model.")

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 3. Processing Loop
# Using 'magma' colormap for "dark purple to cream" aesthetic
cmap = plt.cm.magma
features = {
    0: {"name": "Land", "ious": [], "counts": [], "color": cmap(0.6)},   # Red/Orange
    1: {"name": "Water", "ious": [], "counts": [], "color": cmap(0.2)},  # Dark Purple
    2: {"name": "Cloud", "ious": [], "counts": [], "color": cmap(0.95)}  # Cream/Yellow
}

# Limit processing to a subset of batches for speed
max_batches = 394
print(f"Processing {max_batches} batches from the training set...")

with torch.no_grad():
    for i, batch in tqdm(enumerate(train_dl), total=max_batches):
        if i >= max_batches:
            break

        inputs = batch["image"].to(device)
        targets = batch["mask"].squeeze(1).to(device)

        logits = model(inputs)
        preds = torch.argmax(logits, dim=1)

        # Iterate over batch
        for j in range(inputs.shape[0]):
            pred = preds[j]
            target = targets[j]
            valid = target != 0 # 0 is NoData

            for cls_idx, info in features.items():
                gt_val = cls_idx + 1
                gt_mask = (target == gt_val)
                pred_mask = (pred == cls_idx) & valid

                intersection = (gt_mask & pred_mask).sum().item()
                union = (gt_mask | pred_mask).sum().item()

                if union > 0:
                    info["ious"].append(intersection / union)
                    info["counts"].append(gt_mask.sum().item())

# 4. Plotting
plt.figure(figsize=(6, 12)) # Vertical aspect ratio

for cls_idx, info in features.items():
    plt.scatter(info["counts"], info["ious"], alpha=0.6, s=15,
               c=[info["color"]], label=info["name"], edgecolors='none') # c requires list for single color

plt.title("IoU vs Pixel Count for All Classes")
plt.ylabel("IoU")
plt.xlabel("Number of Pixels (Ground Truth)")
plt.legend(loc="lower right")
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()